# Voltametria de Corrente Amostrada

A voltametria de corrente amostrada consiste em uma sequência de saltos de potencial, nos quais a corrente é medida após um tempo de amostragem pré-determinado $\tau_s$. As equações diferenciais, condições iniciais e condições de contorno são idênticas às do modelo de degrau de potencial reversível — segunda lei de Fick adimensionalizada com equilíbrio de Nernst na superfície do eletrodo.

A diferença está na forma como o problema é resolvido: em vez de um único salto de potencial, a solução é obtida para múltiplos saltos entre um mesmo $E_i$ e distintos valores de $E_f$. Em cada potencial aplicado, a corrente é avaliada no instante $t = 1$ (correspondente a $\tau_s$) e normalizada pela corrente limite de Cottrell. Ao final, constrói-se a curva $i(E)$, que representa a resposta do sistema a cada degrau de potencial aplicado.

## Bibliotecas:

In [1]:
import pybamm
import numpy as np
import matplotlib.pyplot as plt
from scipy import special
from ipywidgets import Text, Button, HBox, VBox, Output
from IPython.display import display, clear_output

## Definindo o Modelo:

In [2]:
model = pybamm.BaseModel()

concentration_o = pybamm.Variable("Concentração de O", domain="electrolyte")
concentration_r = pybamm.Variable("Concentração de R", domain="electrolyte")

overpotential = pybamm.Parameter("Sobrepotencial Aplicado [V]")
faraday       = pybamm.Parameter("Constante de Faraday [C.mol-1]")
gas_constant  = pybamm.Parameter("Constante dos Gases [J.K-1.mol-1]")
temperature   = pybamm.Parameter("Temperatura [K]")

flux_o = -pybamm.grad(concentration_o)   # fluxo difusional de O
flux_r = -pybamm.grad(concentration_r)   # fluxo difusional de R

model.rhs = {
    concentration_o: -pybamm.div(flux_o),  # lei de Fick para O (eq. 1.0a)
    concentration_r: -pybamm.div(flux_r),  # lei de Fick para R (eq. 1.0b)
}

# condições iniciais
model.initial_conditions = {
    concentration_o: pybamm.Scalar(1),  # c_O(x,0) = 1: O uniforme e adimensionalizado
    concentration_r: pybamm.Scalar(0),  # c_R(x,0) = 0: R inicialmente ausente
}

# condições de contorno — Dirichlet
# esquerda (x=0): interface eletrodo/solução — equilíbrio de Nernst
# direita  (x=6): seio da solução            — difusão semi-infinita
nernst_factor = faraday / (gas_constant * temperature)    # f = F/(RT)
theta         = pybamm.exp(nernst_factor * overpotential) # θ = exp(f·η)

model.boundary_conditions = {
    concentration_o: {
        "left":  (theta / (1 + theta), "Dirichlet"),  # eq. 1.3a: c_O(0,t) = θ/(1+θ)
        "right": (pybamm.Scalar(1),    "Dirichlet"),  # eq. 1.2a: c_O(∞,t) = 1
    },
    concentration_r: {
        "left":  (1 / (1 + theta),    "Dirichlet"),  # eq. 1.3b: c_R(0,t) = 1/(1+θ)
        "right": (pybamm.Scalar(0),    "Dirichlet"),  # eq. 1.2b: c_R(∞,t) = 0
    },
}

model.variables = {
    "Concentração de O": concentration_o,
    "Concentração de R": concentration_r,
    "Fluxo de O":        flux_o,
    "Fluxo de R":        flux_r,
}

param = pybamm.ParameterValues(
    {
        "Sobrepotencial Aplicado [V]":        "[input]",
        "Constante de Faraday [C.mol-1]":     96485.3,
        "Constante dos Gases [J.K-1.mol-1]":  8.31446,
        "Temperatura [K]":                    298.15,
    }
)

## Geometria:

In [3]:
# ── Geometria  ─────────────────────────────────────────────────────────

x_variable = pybamm.SpatialVariable(
    "x", domain=["electrolyte"], coord_sys="cartesian"
)

geometry = {
    "electrolyte": {x_variable: {"min": pybamm.Scalar(0), "max": pybamm.Scalar(6)}}
}

## Malha:

In [4]:
submesh_types = {
    "electrolyte": pybamm.MeshGenerator(
        pybamm.Exponential1DSubMesh,
        submesh_params={
            "side": "left",
            "stretch": 5,
        },
    )
}
variable_points = {x_variable: 400}
mesh            = pybamm.Mesh(geometry, submesh_types, variable_points)

## Discretização:

In [5]:

# ── Discretização ─────────────────────────────────────────────────────────────

spatial_methods = {"electrolyte": pybamm.FiniteVolume()}
discretisation  = pybamm.Discretisation(mesh, spatial_methods)

param.process_model(model)
param.process_geometry(geometry)
discretisation.process_model(model)

## Solver

In [6]:
solver = pybamm.IDAKLUSolver()

## Gráfico Interativo:

In [7]:
NERNST_FACTOR = 38.9217  # f = F/(RT) a 298.15 K [V⁻¹]

# pontos de tempo: a corrente é amostrada apenas em t=1, mas o solver precisa
# de uma malha temporal para convergir corretamente até esse instante.
# reduzir abaixo de 200 pode comprometer a precisão — ajuste com cautela.
time = np.linspace(1e-5, 1, 200)

output = Output()

def plot(initial_potential_str, final_potential_str, standard_potential_str, number_of_points_str):
    with output:
        clear_output(wait=True)

        # validação das entradas
        try:
            initial_potential  = float(initial_potential_str)
            final_potential    = float(final_potential_str)
            standard_potential = float(standard_potential_str)
            number_of_points   = int(number_of_points_str)
        except ValueError:
            print("Por favor, insira valores numéricos válidos.")
            return

        if initial_potential >= final_potential:
            print("O potencial inicial deve ser menor que o potencial final.")
            return

        if number_of_points < 2:
            print("O número de pontos deve ser maior que 1.")
            return

        # potenciais dos saltos — linspace evita imprecisões do arange com floats
        potentials      = np.linspace(initial_potential, final_potential, number_of_points)
        sampled_current = []

        for applied_potential in potentials:
            solution = solver.solve(
                model, time,
                inputs={"Sobrepotencial Aplicado [V]": applied_potential - standard_potential}
            )
            flux_o_solution = solution["Fluxo de O"]
            # corrente amostrada em t=1, normalizada pela corrente limite de Cottrell
            sampled_current.append(flux_o_solution(1, x=0) * np.sqrt(np.pi))

        fig, ax = plt.subplots(figsize=(6.5, 4))

        ax.plot(potentials, sampled_current, "r-", linewidth=1.5)
        ax.set_xlabel(r"$E$ / V")
        ax.set_ylabel(r"$i$")
        ax.set_xlim([initial_potential, final_potential])
        ax.set_title("Voltametria de Corrente Amostrada")

        plt.tight_layout()
        display(fig)
        plt.close(fig)


# ── Widgets ───────────────────────────────────────────────────────────────────

field_initial_potential  = Text(
    value="-0.5",
    description="$E_i$ [V]:",
    style={"description_width": "initial"},
)

field_final_potential = Text(
    value="0.5",
    description="$E_f$ [V]:",
    style={"description_width": "initial"},
)

field_standard_potential = Text(
    value="0",
    description="$E^0$ [V]:",
    style={"description_width": "initial"},
)

field_number_of_points = Text(
    value="30",
    description="Nº de Pontos:",
    style={"description_width": "initial"},
)

button = Button(description="Recalcular", button_style="success")

def on_click(b):
    plot(
        field_initial_potential.value,
        field_final_potential.value,
        field_standard_potential.value,
        field_number_of_points.value,
    )

button.on_click(on_click)

interface = VBox([
    HBox([field_initial_potential, field_final_potential]),
    HBox([field_standard_potential, field_number_of_points, button]),
    output,
])
display(interface)

# exibe o gráfico inicial com os valores padrão
plot(
    field_initial_potential.value,
    field_final_potential.value,
    field_standard_potential.value,
    field_number_of_points.value,
)

O gráfico apresenta a corrente amostrada em função do potencial aplicado $E$. A curva tem formato sigmoidal, característico de um processo reversível controlado por difusão.

Para potenciais muito negativos em relação a $E^0$, a concentração superficial de $O$ tende a zero e a corrente atinge um patamar — a corrente de difusão limite. Para potenciais muito positivos, a redução de $O$ não ocorre e a corrente tende a zero. Em sistemas reversíveis a inflexão da curva ocorre exatamente no valor de $E^0$, onde $\theta = 1$, isto é, as concentrações superficiais de $O$ e $R$ são iguais.

O número de pontos controla a resolução da curva — quanto maior, mais suave o traçado, porém com maior tempo de cálculo, pois cada ponto exige uma simulação completa.